In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 72
from IPython.display import Markdown
import numpy as np
import pandas as pd
import os
import json
from sklearn.metrics.pairwise import cosine_similarity


MODEL_PATH= 'EP_models/'
os.environ['HF_HOME'] = MODEL_PATH  # before import transformers
os.environ['HF_DATASETS_OFFLINE']= '1'

In [2]:
import torch
import transformers
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

# filter warnings
import warnings
transformers.logging.set_verbosity_error()
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

Device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f'PyTorch version= {torch.__version__}')
print(f'transformers version= {transformers.__version__}')
print(f'CUDA available= {torch.cuda.is_available()}')

PyTorch version= 2.10.0
transformers version= 5.0.0
CUDA available= False


In [ ]:
# squad_data['data'][200]['paragraphs'][0]['qas']# explore SQuAD training data structure
with open('EP_datasets/squad-train-v2.0.json', 'r') as f:
    squad_data = json.load(f)

n_titles = len(squad_data['data'])
print(f'There are {n_titles} titles/topics in SQuAD v2.0 training dataset.\n')

title_idx = 200
prg_idx = 2
current_title = squad_data['data'][title_idx]['title']
n_paragraphs = len(squad_data['data'][title_idx]['paragraphs'])
n_qas = len(squad_data['data'][title_idx]['paragraphs'][prg_idx]['qas'])
print(f"Title with index #{title_idx} is {squad_data['data'][title_idx]['title']}.")
print(f"Title {current_title} has {n_paragraphs} paragraphs. Paragraph with index {prg_idx} has {n_qas} question-answer pairs.")

# docs = [entry['context'] for entry in squad_data['data'][0]['paragraphs']]
# squad_data['data']

There are 442 titles/topics in SQuAD v2.0 training dataset.

Title with index #200 is Florida.
Title Florida has 35 paragraphs. Paragraph with index 2 has 10 question-answer pairs.


In [ ]:
# squad_data['data'][200]['paragraphs'][0]['qas']
squad_data['data'][200]['paragraphs'][0]['context']

In [4]:
# squad_data['data'][200]['paragraphs'][0]['qas']
squad_data['data'][200]['paragraphs'][0]['context']

'Florida i/ˈflɒrɪdə/ (Spanish for "flowery land") is a state located in the southeastern region of the United States. The state is bordered to the west by the Gulf of Mexico, to the north by Alabama and Georgia, to the east by the Atlantic Ocean, and to the south by the Straits of Florida and the sovereign state of Cuba. Florida is the 22nd most extensive, the 3rd most populous, and the 8th most densely populated of the United States. Jacksonville is the most populous city in Florida, and the largest city by area in the contiguous United States. The Miami metropolitan area is the eighth-largest metropolitan area in the United States. Tallahassee is the state capital.'

In [3]:
sqdata = {"topic": [], "context": []}

for i in range(n_titles):
    n_paragraphs = len(squad_data['data'][i]['paragraphs'])
    for j in range(n_paragraphs):
        sqdata["topic"].append(squad_data['data'][i]['title'])
        sqdata["context"].append(squad_data['data'][i]['paragraphs'][j]['context'])
    

df = pd.DataFrame(sqdata)

# sanity
print(f'Shape of df: {df.shape}')
df.head()

NameError: name 'n_titles' is not defined

In [1]:
class RAGLLM:
    def __init__(self, external_data):
        self.st_model = SentenceTransformer('all-MiniLM-L6-v2')  # sentence transformer is used for embeddings
        
        # external data
        self.external_data = external_data
        self.titles = self.external_data['topic'].unique()
        self.title_embeddings = self.st_model.encode(self.titles.tolist())  # embed each topic/title

        # LLM model
        self.generator = pipeline('text-generation', model='meta-llama/Llama-3.2-1B-Instruct',
                                  max_length=4096, device=Device)  # utilize GPU in this example

    def generate_answer(self, user_question, include_context):
        # Step 1: Find the top-k best matching topic
        question_embedding = self.st_model.encode([user_question])
        title_similarities = cosine_similarity(question_embedding, self.title_embeddings)[0]

        k = 5
        top_k_indices = np.argsort(title_similarities)[-k:][::-1]
        top_k_titles = [str(self.titles[i]) for i in top_k_indices]

        print(f"Top-{k} topics: {top_k_titles}\n")

        # # Step 2: Filter contexts for the top-k best topics
        topic_contexts = self.external_data[self.external_data['topic'].isin(top_k_titles)]['context'].tolist()

        # Step 3: Vectorize each context under the best topic and find the best matching context
        context_embeddings = self.st_model.encode(topic_contexts)
        context_similarities = cosine_similarity(question_embedding, context_embeddings)
        best_context_index = np.argmax(context_similarities)
        best_context = topic_contexts[best_context_index]
        print(f"The best paragraph: {best_context}\n")

        # Step 4: Generate the answer using LLM with retrieved context
        prompt = f"Context: {best_context}\n\nQuestion: {user_question}\nAnswer:"
        response = self.generator(prompt, max_length=300)
        answer_text = response[0]['generated_text']
        answer = answer_text.split("Answer: ")[-1] 
        return answer
    
    def generate_answer_with_no_context(self, user_question):
        # Step 4: Generate the answer using LLM with retrieved context
        prompt = f"Question: {user_question}\nAnswer:"
        response = self.generator(prompt, max_length=300)
        answer_text = response[0]['generated_text']
        answer = answer_text.split("Answer: ")[-1] 
        return answer

In [2]:
query = "How does Beyoncé describe herself?"
ragllm = RAGLLM(df)
output = ragllm.generate_answer(query)
for part in output.split("\n\n"):
    print(part)

output = ragllm.generate_answer_with_no_context(query)
for part in output.split("\n\n"):
    print(part)

NameError: name 'df' is not defined

In [7]:
query = "Who is the US president in 2012?"
ragllm = RAGLLM(df)
output = ragllm.generate_answer(query)
for part in output.split("\n\n"):
    print(part)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

The best paragraph: The Times occasionally makes endorsements for foreign elections. In November 2012, it endorsed a second term for Barack Obama although it also expressed reservations about his foreign policy.

Yes, he is the US president in 2012.


In [23]:
query = "What is the largest state in USA by area?"
ragllm = RAGLLM(df)
output = ragllm.generate_answer(query)
for part in output.split("\n\n"):
    print(part)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Top-5 topics: ['Geography_of_the_United_States', '51st_state', 'Alaska', 'Montana', 'Florida']

The best paragraph: With a total area of 147,040 square miles (380,800 km2), Montana is slightly larger than Japan. It is the fourth largest state in the United States after Alaska, Texas, and California; the largest landlocked U.S. state; and the 56th largest national state/province subdivision in the world. To the north, Montana shares a 545-mile (877 km) border with three Canadian provinces: British Columbia, Alberta, and Saskatchewan, the only state to do so. It borders North Dakota and South Dakota to the east, Wyoming to the south and Idaho to the west and southwest.

Alaska
Explanation: Alaska is the largest state in the United States by area.


In [ ]:
query = "Which state is larger by area in the USA: Alaska or Texas?"
ragllm = RAGLLM(df)
output = ragllm.generate_answer(query)
for part in output.split("\n\n"):
    print(part)

In [25]:
query = "What is the population of Florida in 2025?"
ragllm = RAGLLM(df)
output = ragllm.generate_answer(query)
for part in output.split("\n\n"):
    print(part)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Top-5 topics: ['Florida', 'Miami', 'Charleston,_South_Carolina', 'Financial_crisis_of_2007%E2%80%9308', 'United_Nations_Population_Fund']

The best paragraph: The United States Census Bureau estimates that the population of Florida was 20,271,272 on July 1, 2015, a 7.82% increase since the 2010 United States Census. The population of Florida in the 2010 census was 18,801,310. Florida was the seventh fastest-growing state in the U.S. in the 12-month period ending July 1, 2012. In 2010, the center of population of Florida was located between Fort Meade and Frostproof. The center of population has moved less than 5 miles (8 km) to the east and approximately 1 mile (1.6 km) to the north between 1980 and 2010 and has been located in Polk County since the 1960 census. The population exceeded 19.7 million by December 2014, surpassing the population of the state of New York for the first time.

Not found Context: The population of Florida in 2025 is unknown because the United States Census Bur

In [27]:
query = "Where is Mount Rushmore?"
ragllm = RAGLLM(df)
output = ragllm.generate_answer(query)
for part in output.split("\n\n"):
    print(part)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Top-5 topics: ['Appalachian_Mountains', 'Montana', 'Somerset', 'Alps', 'Egypt']

The best paragraph: In Pennsylvania, there are over sixty summits that rise over 2,500 ft (800 m); the summits of Mount Davis and Blue Knob rise over 3,000 ft (900 m). In Maryland, Eagle Rock and Dans Mountain are conspicuous points reaching 3,162 ft (964 m) and 2,882 ft (878 m) respectively. On the same side of the Great Valley, south of the Potomac, are the Pinnacle 3,007 feet (917 m) and Pidgeon Roost 3,400 ft (1,000 m). In West Virginia, more than 150 peaks rise above 4,000 ft (1,200 m), including Spruce Knob 4,863 ft (1,482 m), the highest point in the Allegheny Mountains. A number of other points in the state rise above 4,800 ft (1,500 m). Snowshoe Mountain at Thorny Flat 4,848 ft (1,478 m) and Bald Knob 4,842 ft (1,476 m) are among the more notable peaks in West Virginia.

Not found Context: Mount Rushmore is located in the Black Hills of South Dakota, which is in the western part of the state.


In [ ]:
class LLaMAQA:
    def __init__(self, model_name='meta-llama/Llama-3.2-1B-Instruct'):
        self.model_name = model_name
        print(f"Loading model '{model_name}'...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)

    def ask_question(self, question, max_length=50, temperature=0.7, top_p=0.9, top_k=50):
        """
        Ask a question and get a response from the model.
        Parameters:
            question (str): The question to ask.
            max_length (int): Maximum length of the response.
            temperature (float): Sampling temperature for randomness.
            top_p (float): Nucleus sampling parameter.
            top_k (int): Top-k sampling parameter.
        Returns:
            str: The model's response.
        """
        inputs = self.tokenizer(question, return_tensors="pt")
        
        # generate the response
        outputs = self.model.generate(
            inputs["input_ids"],
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            do_sample=True
        )
        # decode and return the response
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
llama_qa = LLaMAQA()

query = "What does 'Florida' mean in Spanish?"
response = llama_qa.ask_question(query)
print("Q:", query)
print("A:", response)